## <span style="color:#4375c7">Data and AI in Economics</span>
***
*The course material is for educational purposes only. Nothing herein should be taken as an investment advice or offers any opinion regarding the suitability of any security. For more information about this course, please contact us.*
***
## 6. Solution — Transformers & GPT

This notebook contains the complete solutions to the exercise tasks.

In [ ]:
!pip install -q -r https://raw.githubusercontent.com/firrm/DAI/main/requirements_colab.txt

import numpy as np
import matplotlib.pyplot as plt
import math
import urllib.request

## 2. Hands-on session <a id='ho'></a>

### Exercise 1 — Character-level tokenizer

Build a character-level tokenizer for a list of first names.

A tokenizer maps every unique character to a unique integer id and provides:
- `encode(name)` → list of ints (wrap in BOS tokens)
- `decode(ids)` → string (skip BOS tokens)

**Tasks:**
1. Extract all unique characters from `names` and sort them.
2. Build `stoi` (char → id) and `itos` (id → char) dictionaries. Assign id 0 to the special `BOS` token and ids 1–26 to the 26 lowercase letters.
3. Implement `encode(name)` and `decode(ids)`.
4. Verify the round-trip: `decode(encode(name)) == name` for several names.
5. Print the number of unique tokens and the vocabulary.

In [ ]:
# Fetch the names dataset
url = "https://raw.githubusercontent.com/karpathy/micrograd/master/names.txt"
try:
    with urllib.request.urlopen(url) as r:
        names_text = r.read().decode()
except Exception:
    names_text = "emma\nolivia\nava\nisabella\nsophia\ncharlotte\nmia\namelia\nharper\nevelyn\n"

names = [n.strip() for n in names_text.strip().split("\n") if n.strip()]
print(f"Loaded {len(names):,} names. First 5: {names[:5]}")

In [ ]:
## Task 1: Build the vocabulary

chars = sorted(set("".join(names)))
vocab = ["BOS"] + chars
vocab_size = len(vocab)

stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}

BOS = stoi["BOS"]

print(f"Vocabulary ({vocab_size} tokens):", vocab)

In [ ]:
## Task 2: Implement encode() and decode()

def encode(name):
    """Encode a name as a list of integer ids, wrapped in BOS tokens."""
    return [BOS] + [stoi[c] for c in name] + [BOS]

def decode(ids):
    """Decode a list of integer ids back to a string (ignore BOS tokens)."""
    return "".join(itos[i] for i in ids if i != BOS)

# Verify round-trip
for test_name in ["emma", "olivia", "ava"]:
    assert decode(encode(test_name)) == test_name, f"Round-trip failed for '{test_name}'"
    print(f"encode('{test_name}') = {encode(test_name)}")
print("✓ All round-trips OK")

### Exercise 2 — Softmax & cross-entropy loss

Implement the two core building blocks used to turn model outputs into a training signal.

**Tasks:**
1. Implement `softmax(logits)` with numerical stability (subtract max before exponentiation).
2. Implement `cross_entropy_loss(probs, target_id)` using $-\log(p_{\text{target}})$.
3. Verify: `softmax` output is non-negative and sums to 1.
4. Plot the loss $-\log(p)$ as a function of $p \in (0, 1)$ and mark the random-baseline loss $-\log(1/27)$.

In [ ]:
## Task 1 & 2: Implement softmax and cross-entropy

def softmax(logits):
    """Numerically stable softmax. Input: 1-D array. Output: probability array."""
    logits = np.array(logits, dtype=float)
    logits -= logits.max()
    exp_l = np.exp(logits)
    return exp_l / exp_l.sum()

def cross_entropy_loss(probs, target_id):
    """Cross-entropy loss for a single prediction."""
    return -np.log(probs[target_id] + 1e-12)

# Verify
logits_test = np.array([2.0, 1.0, 0.1, -1.0, 3.0])
probs_test  = softmax(logits_test)
assert abs(probs_test.sum() - 1.0) < 1e-9, "Probs must sum to 1"
assert (probs_test >= 0).all(), "Probs must be non-negative"
print("softmax probs:", np.round(probs_test, 4), "  sum:", probs_test.sum().round(6))

loss_high_conf = cross_entropy_loss(np.array([0.01, 0.99]), 1)
loss_low_conf  = cross_entropy_loss(np.array([0.99, 0.01]), 1)
print(f"Loss when model is 99% confident and correct:   {loss_high_conf:.4f}")
print(f"Loss when model is  1% confident and correct:   {loss_low_conf:.4f}")

In [ ]:
## Task 3: Plot the loss curve and mark the random baseline

p_vals = np.linspace(0.01, 0.99, 200)
loss_vals = -np.log(p_vals)

plt.figure(figsize=(6, 3.5))
plt.plot(p_vals, loss_vals, color="#4375c7", lw=2, label="$-\\log(p)$")
plt.axhline(-np.log(1/27), color="gray", ls="--",
            label=f"Random baseline (27 tokens): {-np.log(1/27):.2f}")
plt.xlabel("Probability of correct token $p$")
plt.ylabel("Loss  $-\\log(p)$")
plt.title("Cross-entropy loss")
plt.legend(); plt.tight_layout(); plt.show()

### Exercise 3 — Scaled dot-product attention

Implement the core of the Transformer: scaled dot-product attention with a causal mask.

Given queries $Q$, keys $K$, values $V$ (all shape `(T, d_k)`):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

where $M$ is a causal mask with $-\infty$ in the upper triangle (future positions).

**Tasks:**
1. Implement `attention(Q, K, V)` — compute scaled scores, apply causal mask, softmax, weight `V`.
2. Create random `Q`, `K`, `V` matrices (shape `(6, 4)`) and call your function.
3. Visualise the attention weight matrix as a heatmap. Verify the upper triangle is zero.

In [ ]:
## Task 1: Implement scaled dot-product attention with causal mask

def attention(Q, K, V):
    """
    Scaled dot-product attention with causal masking.
    Q, K, V: arrays of shape (T, d_k)
    Returns: output of shape (T, d_k)
    """
    T, dk = Q.shape
    scores  = Q @ K.T / math.sqrt(dk)
    mask    = np.triu(np.full((T, T), -np.inf), k=1)
    scores  = scores + mask
    weights = np.apply_along_axis(softmax, axis=1, arr=scores)
    return weights @ V

# Test
np.random.seed(7)
T, dk = 6, 4
Q_test = np.random.randn(T, dk)
K_test = np.random.randn(T, dk)
V_test = np.random.randn(T, dk)
out_test = attention(Q_test, K_test, V_test)
print("Attention output shape:", out_test.shape)  # expected: (6, 4)

In [ ]:
## Task 2: Visualise the attention weight matrix

scores_viz  = Q_test @ K_test.T / math.sqrt(dk)
mask_viz    = np.triu(np.full((T, T), -np.inf), k=1)
weights_viz = np.apply_along_axis(softmax, 1, scores_viz + mask_viz)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights_viz, cmap="Blues", vmin=0, vmax=1)
ax.set_title("Attention weights (causal mask applied)")
ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

assert np.allclose(np.triu(weights_viz, k=1), 0), "Causal mask not applied correctly!"
print("✓ Causal mask verified: no future tokens are attended to")

### Exercise 4 — Keras transformer on economics text

Build and train a small character-level causal GPT using Keras on a short FOMC statement excerpt. Then generate text at two different temperatures and observe the effect.

**Tasks:**
1. Tokenise the provided `corpus` string at the character level.
2. Build sliding-window training sequences of length `SEQ_LEN = 40`.
3. Implement `causal_gpt()` using `keras.layers.MultiHeadAttention` with `use_causal_mask=True`.
4. Compile with Adam and SparseCategoricalCrossentropy; train for 200 epochs.
5. Implement `generate_text()` and sample at temperatures 0.5 and 1.0. Compare the results.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
import tensorflow as tf

corpus = """
The Federal Open Market Committee decided to maintain the target range for the federal funds rate.
The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.
In support of these goals, the Committee decided to raise the target range for the federal funds rate.
Recent indicators suggest that economic activity has continued to expand at a modest pace.
Job gains have been robust in recent months, and the unemployment rate has remained low.
Inflation remains elevated, reflecting supply and demand imbalances related to the pandemic.
The Committee is highly attentive to inflation risks and is committed to returning inflation to its 2 percent objective.
The pace of future increases in the target range will take into account the cumulative tightening of monetary policy.
""".strip()

In [ ]:
## Task 1: Character-level tokenisation of the corpus

chars_corp     = sorted(set(corpus))
vocab_c        = len(chars_corp)
s2i_c          = {c: i for i, c in enumerate(chars_corp)}
i2s_c          = {i: c for c, i in s2i_c.items()}
encoded_corpus = [s2i_c[c] for c in corpus]

print(f"Corpus: {len(corpus)} characters, {vocab_c} unique chars")

In [ ]:
## Task 2: Build sliding-window training sequences

SEQ_LEN = 40

def make_sequences(enc, seq_len):
    """Return input array X and target array y for next-character prediction."""
    X, y = [], []
    for i in range(len(enc) - seq_len):
        X.append(enc[i : i + seq_len])
        y.append(enc[i + 1 : i + seq_len + 1])
    return np.array(X), np.array(y)

X, y = make_sequences(encoded_corpus, SEQ_LEN)
print(f"X shape: {X.shape},  y shape: {y.shape}")

In [ ]:
## Task 3: Implement the Keras causal GPT

D_MODEL  = 64
N_HEADS  = 4
D_FF     = 256
N_LAYERS = 2

def causal_gpt(vocab_size, seq_len, d_model, n_heads, d_ff, n_layers):
    inputs = keras.Input(shape=(seq_len,), dtype="int32")

    tok_emb = layers.Embedding(vocab_size, d_model)(inputs)
    pos_idx = tf.range(seq_len)
    pos_emb = layers.Embedding(seq_len, d_model)(pos_idx)
    x = tok_emb + pos_emb

    for _ in range(n_layers):
        attn_out = layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model // n_heads, use_bias=False
        )(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + attn_out)

        ff = layers.Dense(d_ff, activation="relu")(x)
        ff = layers.Dense(d_model)(ff)
        x = layers.LayerNormalization()(x + ff)

    logits = layers.Dense(vocab_size)(x)
    return keras.Model(inputs, logits)

model_keras = causal_gpt(vocab_c, SEQ_LEN, D_MODEL, N_HEADS, D_FF, N_LAYERS)
model_keras.summary()

In [ ]:
## Task 4: Compile and train

model_keras.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
history = model_keras.fit(X, y, batch_size=32, epochs=200, verbose=0)

plt.figure(figsize=(7, 3))
plt.plot(history.history["loss"], color="#4375c7")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training loss"); plt.tight_layout(); plt.show()

In [ ]:
## Task 5: Implement generate_text() and compare temperatures

def generate_text(model, seed_text, temperature=1.0, n_chars=150):
    """Autoregressively generate text from the trained Keras model."""
    rng = np.random.default_rng(0)
    context = [s2i_c.get(c, 0) for c in seed_text[-SEQ_LEN:]]
    result  = list(seed_text)
    for _ in range(n_chars):
        padded = context[-SEQ_LEN:]
        if len(padded) < SEQ_LEN:
            padded = [0] * (SEQ_LEN - len(padded)) + padded
        x_in   = np.array([padded])
        logits = model.predict(x_in, verbose=0)[0, -1]
        logits = logits / temperature
        probs  = softmax(logits)
        next_id = rng.choice(len(probs), p=probs)
        result.append(i2s_c[next_id])
        context.append(next_id)
    return "".join(result)

print("=== temperature = 0.5 (more focused) ===")
print(generate_text(model_keras, "The Committee", temperature=0.5))
print()
print("=== temperature = 1.0 (more diverse) ===")
print(generate_text(model_keras, "The Committee", temperature=1.0))